In [21]:
import sys
import os
import importlib
import src.data_preprocessing as dp

sys.path.insert(0, os.path.abspath('../..'))
importlib.reload(dp)

from src.data_preprocessing import *

In [22]:
df = load_dataset("../../data/raw/cardio_train.csv", sep=";")
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [23]:
df.duplicated().sum()

np.int64(0)

In [24]:
df["age"] = df["age"].apply(process_age)
df["gender"] = df["gender"].map({1: "male", 2: "female"})
df = pd.get_dummies(df, columns=["gender"], drop_first=True, dtype=int)
df.drop(columns=["id"], inplace=True)
df.head()

,age,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,gender_male
0,50,168,62.0,110,80,1,1,0,0,1,0,0
1,55,156,85.0,140,90,3,1,0,0,1,1,1
2,51,165,64.0,130,70,3,1,0,0,0,1,1
3,48,169,82.0,150,100,1,1,0,0,1,1,0
4,47,156,56.0,100,60,1,1,0,0,0,0,1


In [25]:
df.isna().sum()

age            0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
gender_male    0
dtype: int64

In [26]:
print("Valeurs aberrantes - Taille < 120 cm :", len(df[df["height"] < 120]))
print("Valeurs aberrantes - Taille > 220 cm :", len(df[df["height"] > 220]))
print("-----------------------------")
print("Valeurs aberrantes - Poids < 30 kg :", len(df[df["weight"] < 30]))
print("Valeurs aberrantes - Poids > 200 kg :", len(df[df["weight"] > 200]))
print("-----------------------------")
print("Valeurs aberrantes - Pression systolique <= 0 :", len(df[df["ap_hi"] <= 0]))
print("Valeurs aberrantes - Pression diastolique <= 0 :", len(df[df["ap_lo"] <= 0]))
print("-----------------------------")
print("Valeurs aberrantes - Pression systolique > 250 mmHg :", len(df[df["ap_hi"] > 250]))
print("Valeurs aberrantes - Pression diastolique > 200 mmHg :", len(df[df["ap_lo"] > 200]))
print("-----------------------------")
print("Incohérences physiologiques (ap_hi <= ap_lo) :", len(df[df["ap_hi"] <= df["ap_lo"]]))

Valeurs aberrantes - Taille < 120 cm : 52
Valeurs aberrantes - Taille > 220 cm : 1
-----------------------------
Valeurs aberrantes - Poids < 30 kg : 7
Valeurs aberrantes - Poids > 200 kg : 0
-----------------------------
Valeurs aberrantes - Pression systolique <= 0 : 7
Valeurs aberrantes - Pression diastolique <= 0 : 22
-----------------------------
Valeurs aberrantes - Pression systolique > 250 mmHg : 40
Valeurs aberrantes - Pression diastolique > 200 mmHg : 953
-----------------------------
Incohérences physiologiques (ap_hi <= ap_lo) : 1236


In [27]:
initial_rows = len(df)

df_clean = df[
    (df["ap_hi"] > 0) &
    (df["ap_lo"] > 0) &
    (df["ap_hi"] > df["ap_lo"]) &
    (df["ap_hi"] <= 250) &
    (df["ap_lo"] <= 200)
].copy()

print(f"Nombre de lignes avant nettoyage : {initial_rows}")
print(f"Nombre de lignes après nettoyage : {len(df_clean)}")
print(f"Nombre de lignes supprimées : {initial_rows - len(df_clean)}")
print(f"Pourcentage supprimé : {((initial_rows - len(df_clean)) / initial_rows) * 100:.2f}%")

Nombre de lignes avant nettoyage : 70000
Nombre de lignes après nettoyage : 68709
Nombre de lignes supprimées : 1291
Pourcentage supprimé : 1.84%


In [29]:
save_dataset(df_clean, "../../data/processed/cardio_train.csv")